# 🌿 Pineapple Plant Analysis System
**Three-task model trained on a single T4 GPU**

| Task | Output | Loss |
|------|--------|------|
| Health classification | healthy / nitrogen_deficiency / water_stress | CrossEntropy (weighted) |
| Growth stage | M1–M12 | CrossEntropy (weighted) |
| Width estimation | cm (positive) | HuberLoss (masked) |

Total loss = **0.4 × L_health + 0.4 × L_month + 0.2 × L_width**

In [ ]:
# ── Cell 1 — Check GPU ────────────────────────────────────────────────
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print(f'CUDA available: {torch.cuda.is_available()}  |  Device: {gpu_name}')
if not torch.cuda.is_available():
    print('⚠️  No GPU detected. Training will be slow. Enable GPU: Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2 — Install dependencies ────────────────────────────────────
!pip install -q torch torchvision opencv-python-headless scikit-learn scipy matplotlib tqdm

In [ ]:
# ── Cell 3 — Upload / clone code ─────────────────────────────────────
# OPTION A: Clone from GitHub (recommended)
# !git clone https://github.com/YOUR_USER/pineapple_pipeline.git /content/pineapple_pipeline

# OPTION B: Upload zip
# from google.colab import files
# up = files.upload()   # upload pineapple_pipeline.zip
# !unzip -q pineapple_pipeline.zip -d /content/

import sys, os
sys.path.insert(0, '/content/pineapple_pipeline')
print('Code path added.')

In [ ]:
# ── Cell 4 — Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Adjust these paths ────────────────────────────────────────────────
DATASET_ROOT = '/content/drive/MyDrive/PineappleDataset'   # root with M1..M12 dirs
WIDTH_CSV    = '/content/drive/MyDrive/PineappleDataset/width_labels.csv'  # optional
RUN_DIR      = '/content/drive/MyDrive/pineapple_runs/exp01'
print(f'Dataset root : {DATASET_ROOT}')
print(f'Run output   : {RUN_DIR}')

In [ ]:
# ── Cell 5 — Verify dataset structure ────────────────────────────────
import os
from pathlib import Path

root = Path(DATASET_ROOT)
print(f'Exists: {root.exists()}')
total_images = 0
for month_dir in sorted(root.iterdir()):
    if not month_dir.is_dir(): continue
    for health_dir in sorted(month_dir.iterdir()):
        if not health_dir.is_dir(): continue
        imgs = list(health_dir.rglob('*.jpg')) + list(health_dir.rglob('*.png'))
        total_images += len(imgs)
        print(f'  {month_dir.name}/{health_dir.name}: {len(imgs)} images')
print(f'\nTotal images: {total_images}')

In [ ]:
# ── Cell 6 — (Optional) Review width CSV ─────────────────────────────
import pandas as pd, os
if os.path.exists(WIDTH_CSV):
    df = pd.read_csv(WIDTH_CSV)
    print(f'Rows: {len(df)}  |  Columns: {list(df.columns)}')
    print(f'Zero/missing widths: {(df.width_cm == 0).sum()}')
    print(df.describe())
    df.head()
else:
    print('No width CSV found — model will use expected widths from constants as targets.')
    WIDTH_CSV = None

In [ ]:
# ── Cell 7 — Training config ──────────────────────────────────────────
import argparse

args = argparse.Namespace(
    # Data
    data_root      = DATASET_ROOT,
    width_csv      = WIDTH_CSV,
    output_dir     = f'{RUN_DIR}/output',
    checkpoint_dir = f'{RUN_DIR}/checkpoints',

    # Model
    backbone       = 'efficientnet_b0',   # or 'resnet50'
    image_size     = 224,
    dropout        = 0.3,

    # Loss weights  (must sum ≤ 1.0; they don't need to sum to 1)
    alpha          = 0.4,   # health
    beta           = 0.4,   # month
    gamma          = 0.2,   # width
    label_smoothing= 0.05,

    # Training
    batch_size     = 32,
    epochs         = 80,
    lr             = 3e-4,
    weight_decay   = 1e-4,
    patience       = 10,
    freeze_epochs  = 5,     # freeze backbone for first 5 epochs
    seed           = 42,
    num_workers    = 2,     # Colab: keep low
    device         = 'auto',
    strong_aug     = True,
    sampler        = 'joint',
)

print('Config ready:')
for k, v in vars(args).items():
    print(f'  {k:25s} = {v}')

In [ ]:
# ── Cell 8 — Train ────────────────────────────────────────────────────
from pineapple.train import train_main
train_main(args)

In [ ]:
# ── Cell 9 — Plot training curves ────────────────────────────────────
import json, matplotlib.pyplot as plt

with open(f'{args.output_dir}/training_history.json') as f:
    hist = json.load(f)['history']

epochs   = [h['epoch']           for h in hist]
tr_loss  = [h['train_loss']      for h in hist]
vl_loss  = [h['val_loss']        for h in hist]
h_acc    = [h['val_health_acc']  for h in hist]
m_acc    = [h['val_month_acc']   for h in hist]
w_mae    = [h['val_width_mae']   for h in hist]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs, tr_loss, label='Train'); axes[0].plot(epochs, vl_loss, label='Val')
axes[0].set_title('Total Loss'); axes[0].legend()
axes[1].plot(epochs, h_acc, label='Health'); axes[1].plot(epochs, m_acc, label='Month')
axes[1].set_title('Val Accuracy'); axes[1].legend()
axes[2].plot(epochs, w_mae, color='tomato', label='Width MAE (cm)')
axes[2].axhline(2.0, ls='--', c='grey', alpha=0.6, label='Target <2 cm')
axes[2].set_title('Width MAE'); axes[2].legend()
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 10 — Display confusion matrices ──────────────────────────────
from IPython.display import Image as IPImage
IPImage(f'{args.output_dir}/confusion_matrices.png', width=900)

In [ ]:
# ── Cell 11 — View test metrics ───────────────────────────────────────
import json
with open(f'{args.output_dir}/test_metrics.json') as f:
    tm = json.load(f)

print('── Health ──────────────────────────────────────')
print(f"  Accuracy : {tm['health']['accuracy']:.4f}")
print(f"  F1-macro : {tm['health']['f1_macro']:.4f}")
print('── Month ───────────────────────────────────────')
print(f"  Accuracy : {tm['month']['accuracy']:.4f}")
print(f"  F1-macro : {tm['month']['f1_macro']:.4f}")
print(f"  MAE      : {tm['month_mae_months']:.2f} months")
if tm.get('width'):
    print('── Width ───────────────────────────────────────')
    print(f"  MAE (cm) : {tm['width']['mae_cm']:.2f}")
    print(f"  RMSE (cm): {tm['width']['rmse_cm']:.2f}")
    print(f"  R²       : {tm['width']['r2']:.4f}")

In [ ]:
# ── Cell 12 — Single-image inference ─────────────────────────────────
from pineapple import predict
import json

TEST_IMAGE = f'{DATASET_ROOT}/M3/healthy/sample.jpg'   # ← change to real path
CKPT       = f'{args.checkpoint_dir}/best_model.pt'
SAVE_DIR   = f'{RUN_DIR}/sample_inference'

result = predict(
    image_path = TEST_IMAGE,
    model_path = CKPT,
    save_dir   = SAVE_DIR,
)

print(json.dumps(result, indent=2))

In [ ]:
# ── Cell 13 — Display GradCAM overlay ────────────────────────────────
import cv2, matplotlib.pyplot as plt, numpy as np
from pathlib import Path
from IPython.display import display

overlay = cv2.imread(result['gradcam_path'])
overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
orig    = plt.imread(TEST_IMAGE)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(orig);     ax1.set_title('Original');    ax1.axis('off')
ax2.imshow(overlay);  ax2.set_title('Grad-CAM');    ax2.axis('off')
fig.suptitle(
    f"Health: {result['health']} ({result['health_confidence']:.0%})  |  "
    f"Month: {result['month']} ({result['month_confidence']:.0%})  |  "
    f"Width: {result['width_cm']:.1f} cm  |  Stunted: {result['is_stunted']}",
    fontsize=11
)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 14 — Export to ONNX ─────────────────────────────────────────
import argparse
from pineapple.export import export_main

export_args = argparse.Namespace(
    checkpoint = f'{args.checkpoint_dir}/best_model.pt',
    out        = f'{RUN_DIR}/model.onnx',
    format     = 'onnx',
    opset      = 17,
    image_size = args.image_size,
)
export_main(export_args)

In [ ]:
# ── Cell 15 — Batch inference on a folder ────────────────────────────
import pandas as pd
from pathlib import Path
from pineapple import predict

INFER_DIR = f'{DATASET_ROOT}/M6/water_stress'   # ← change
CKPT      = f'{args.checkpoint_dir}/best_model.pt'

records = []
for img_path in sorted(Path(INFER_DIR).rglob('*.jpg'))[:20]:   # cap at 20 for demo
    r = predict(img_path, CKPT)
    records.append({
        'path':         str(img_path),
        'health':       r['health'],
        'health_conf':  r['health_confidence'],
        'month':        r['month'],
        'month_conf':   r['month_confidence'],
        'width_cm':     r['width_cm'],
        'is_stunted':   r['is_stunted'],
    })

df = pd.DataFrame(records)
df.to_csv(f'{RUN_DIR}/batch_predictions.csv', index=False)
print(f'Processed {len(df)} images')
df.head()